# Machine Learning Model Development

### Models: SVM, RandomForest, DecisionTree, XGBoost
### GridSearchCV + SMOTE
### Domains: Food, Fuel, Child

#### Import Libraries

In [3]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore")
import os
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

#### Create Folders For Save The Models

In [4]:
os.makedirs("models/food", exist_ok=True)
os.makedirs("models/fuel", exist_ok=True)
os.makedirs("models/child", exist_ok=True)

#### Load Dataset

In [5]:
df = pd.read_csv("./dataset/feature_engineered_data.csv")

#### Load Feature Lists

In [6]:
with open("./dataset/food_features.json", "r") as f:
    food_features = json.load(f)

with open("./dataset/fuel_features.json", "r") as f:
    fuel_features = json.load(f)

with open("./dataset/child_features.json", "r") as f:
    child_features = json.load(f)

#### Domain Config

In [7]:
domain_map = {
    "food": {"X_cols": food_features, "y_col": "food_security_label"},
    "fuel": {"X_cols": fuel_features, "y_col": "fuel_security_label"},
    "child": {"X_cols": child_features, "y_col": "child_security_label"},
}

#### Models + GridSearch Params

In [8]:
models = {
    "SVM": (
        SVC(),
        {
            "model__C": [1, 3, 5],
            "model__kernel": ["rbf"],
            "model__gamma": ["scale"],
            "model__class_weight": [None, "balanced"],
        },
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=42),
        {
            "model__n_estimators": [200, 400],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_split": [2, 5],
            "model__min_samples_leaf": [1, 2],
            "model__class_weight": [None, "balanced"],
        },
    ),
    "DecisionTree": (
        DecisionTreeClassifier(random_state=42),
        {
            "model__max_depth": [3, 5, 8],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2],
            "model__class_weight": [None, "balanced"],
        },
    ),
    "XGBoost": (
        XGBClassifier(random_state=42, eval_metric="mlogloss"),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [3, 5],
            "model__learning_rate": [0.05, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
        },
    ),
}


#### Prepare Domain Data

In [9]:
def prepare_domain_data(df, cfg):

    X = df[cfg["X_cols"]].copy()
    y = df[cfg["y_col"]].copy()

    mask = y.notna()

    X = X.loc[mask]
    y = y.loc[mask]

    le = LabelEncoder()
    y = le.fit_transform(y.astype(str))

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

#### Build Preprocessor

In [10]:
def build_preprocessor(X_train):

    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

    preprocessor = ColumnTransformer(
        [
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler())
                    ]
                ),
                num_cols
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore"))
                    ]
                ),
                cat_cols
            )
        ]
    )

    return preprocessor

#### Train One Model

In [11]:
def train_one_model(
    domain,
    model_name,
    model,
    param_grid,
    X_train,
    X_test,
    y_train,
    y_test,
    preprocessor,
):

    print("=" * 70)
    print("DOMAIN:", domain.upper())
    print("Training:", model_name)

    pipe = ImbPipeline(
        [("prep", preprocessor), ("smote", SMOTE(random_state=42)), ("model", model)]
    )

    grid = GridSearchCV(
        estimator=pipe, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    # Save model
    save_path = f"models/{domain}/{model_name}.pkl"
    joblib.dump(best_model, save_path)

    print("Saved Model:", save_path)

    # Predictions
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)

    # Metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    train_f1 = f1_score(y_train, y_train_pred, average="weighted")
    test_f1 = f1_score(y_test, y_test_pred, average="weighted")

    result = {
        "Domain": domain,
        "Model": model_name,
        "TrainAccuracy": train_acc,
        "TestAccuracy": test_acc,
        "TrainF1Weighted": train_f1,
        "TestF1Weighted": test_f1,
        "TrainError": 1 - train_acc,
        "TestError": 1 - test_acc,
        "BestCVScoreAccuracy": grid.best_score_,
        "BestParams": str(grid.best_params_),
        "ModelPath": save_path,
    }

    print("Best Test Accuracy:", round(test_acc, 4))

    return result

#### Train All Models for One Domain

In [12]:
def train_all_models_for_domain(domain, cfg, df, models):

    results = []
    X_train, X_test, y_train, y_test = prepare_domain_data(df, cfg)

    preprocessor = build_preprocessor(X_train)
    for model_name, (model, param_grid) in models.items():

        result = train_one_model(
            domain=domain,
            model_name=model_name,
            model=model,
            param_grid=param_grid,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            preprocessor=preprocessor,
        )

        results.append(result)

    return pd.DataFrame(results)

In [13]:
final_results = []

#### Food Module

In [14]:
domain = "food"
cfg = domain_map[domain]

domain_result_df = train_all_models_for_domain(
    domain=domain, cfg=cfg, df=df, models=models
)

final_results.append(domain_result_df)

DOMAIN: FOOD
Training: SVM
Saved Model: models/food/SVM.pkl
Best Test Accuracy: 0.525
DOMAIN: FOOD
Training: RandomForest
Saved Model: models/food/RandomForest.pkl
Best Test Accuracy: 0.5544
DOMAIN: FOOD
Training: DecisionTree
Saved Model: models/food/DecisionTree.pkl
Best Test Accuracy: 0.5221
DOMAIN: FOOD
Training: XGBoost
Saved Model: models/food/XGBoost.pkl
Best Test Accuracy: 0.5853


#### Fuel Module

In [15]:
domain = "fuel"
cfg = domain_map[domain]

domain_result_df = train_all_models_for_domain(
    domain=domain, cfg=cfg, df=df, models=models
)

final_results.append(domain_result_df)

DOMAIN: FUEL
Training: SVM
Saved Model: models/fuel/SVM.pkl
Best Test Accuracy: 0.6338
DOMAIN: FUEL
Training: RandomForest
Saved Model: models/fuel/RandomForest.pkl
Best Test Accuracy: 0.6794
DOMAIN: FUEL
Training: DecisionTree
Saved Model: models/fuel/DecisionTree.pkl
Best Test Accuracy: 0.6647
DOMAIN: FUEL
Training: XGBoost
Saved Model: models/fuel/XGBoost.pkl
Best Test Accuracy: 0.6824


#### Child Module

In [16]:
domain = "child"
cfg = domain_map[domain]

domain_result_df = train_all_models_for_domain(
    domain=domain, cfg=cfg, df=df, models=models
)

final_results.append(domain_result_df)

DOMAIN: CHILD
Training: SVM
Saved Model: models/child/SVM.pkl
Best Test Accuracy: 0.7838
DOMAIN: CHILD
Training: RandomForest
Saved Model: models/child/RandomForest.pkl
Best Test Accuracy: 0.7971
DOMAIN: CHILD
Training: DecisionTree
Saved Model: models/child/DecisionTree.pkl
Best Test Accuracy: 0.7882
DOMAIN: CHILD
Training: XGBoost
Saved Model: models/child/XGBoost.pkl
Best Test Accuracy: 0.7926


#### Save CSV

In [17]:
final_results_df = pd.concat(final_results, ignore_index=True)
final_results_df = final_results_df.sort_values(
    by=["Domain", "TestAccuracy"], ascending=[True, False]
)

In [18]:
final_results_df.to_csv("role3_model_results.csv", index=False)